# Unit explorer

Browse the sorted units of a recording day across its stimulus runs. Pick a run, filter the units by label and by
how selective they are, and step through them. Each unit is shown in **every** run, since one Kilosort sorting covers
the whole day and unit numbers are the same in all runs.

Run all cells, then use the controls at the bottom. The first cell after the settings loads every run (about a
minute each over the network); after that the explorer is instant.

In [ ]:
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from open_rigs_hf_visual import spikes as sp
from open_rigs_hf_visual.bonsai_csv import load_csv_session, trial_conditions, trial_table

## Settings

One entry per stimulus run that has spike sorting: the folder with its CSV files, the protocol (and `start` if it was
run more than once) and its SpikeGLX `_gN` folder. `kilosort_dir` is the day's CatGT processed folder.

On 2026-10-08 the texture session (`_g2`) is not included: its probe data is missing from `Processed_1`.

In [ ]:
stimuli = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\stimuli\20261008\M26031_20261008_1")
ephys = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\ephys\20261008\20261008_1")
kilosort_dir = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\ephys\20261008\Processed_1")

recordings = {
    "DriftingGratings": dict(folder=stimuli, protocol="DriftingGratings", start=None,
                             ephys_root=ephys / "M26031_20261008_1_g1"),
    "DotMotion": dict(folder=stimuli, protocol="DotMotion", start=-1,
                      ephys_root=ephys / "M26031_20261008_1_g0"),
}
window = (-0.5, 1.5)  # time around each onset to plot, in seconds

## Load every run and score every unit

For each run: trials with photodiode onsets, a condition label per trial (`trial_conditions`), and per unit:

- `baseline_rate`, `stim_rate`: mean firing rate (Hz) in the 0.5 s before onset and during the stimulus;
- `response`: `stim_rate - baseline_rate`;
- `selectivity`: one-way ANOVA F statistic of the stimulus-period rate across all conditions, blank included
  (about 1 = no effect of the stimulus). Units that just respond to "stimulus vs blank" score high here;
- `selectivity_stim`: the same, among stimulus conditions only (blank trials left out). This measures tuning, e.g.
  for direction or speed;
- `preferred`: the condition with the highest rate;
- `label` (Kilosort "good" or "mua") and position on the probe (`x`, and `y` = distance from the tip, in µm).

In [ ]:
data = {}
for name, settings in recordings.items():
    session = load_csv_session(settings["folder"], settings["ephys_root"], protocol=settings["protocol"],
                               start=settings["start"], kilosort_dir=kilosort_dir)
    trials = trial_table(session)
    trials["condition"] = trial_conditions(trials)
    stim_window = (0.0, float(np.nanmedian(trials["duration"])))
    spikes = session.spikes
    trains = {u: spikes.spikes(u) for u in spikes.unit_ids(label=None)}
    scores = sp.unit_selectivity(trains, trials["onset"], trials["condition"], stim_window)
    stim = (trials["condition"] != "blank").to_numpy()
    scores["selectivity_stim"] = sp.unit_selectivity(
        trains, trials["onset"][stim], trials["condition"][stim], stim_window)["selectivity"]
    units = spikes.units[["label", "x", "y"]].join(scores, how="inner")
    data[name] = dict(session=session, trials=trials, stim_window=stim_window, units=units, trains=trains)
    print(f"{name}: {len(trials)} trials, {trials['matched'].mean():.0%} matched to the photodiode, "
          f"stimulus {stim_window[1]:.2f} s, {len(units)} units, "
          f"{(units['selectivity'] > 3).sum()} with selectivity > 3")

## Overview

Left: every unit's selectivity in each run (log scale), so units driven by both, one or neither stand out. Right: the
units' positions on the probe, coloured by their best selectivity across runs.

In [ ]:
names = list(data)
all_units = pd.concat({name: d["units"] for name, d in data.items()}, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.2, 1]})
if len(names) >= 2:
    a, b = names[:2]
    colors = all_units[(a, "label")].map({"good": "C3", "mua": "0.6"})
    axes[0].scatter(all_units[(a, "selectivity")], all_units[(b, "selectivity")], s=12, c=colors)
    axes[0].set(xscale="log", yscale="log", xlabel=f"selectivity, {a}", ylabel=f"selectivity, {b}",
                title="Each dot is a unit (red = good, grey = mua)")
    axes[0].axvline(3, color="k", lw=0.5, ls=":")
    axes[0].axhline(3, color="k", lw=0.5, ls=":")
best = np.log10(pd.concat([d["units"]["selectivity"] for d in data.values()], axis=1).max(axis=1).clip(lower=0.1))
first = data[names[0]]["units"]
image = axes[1].scatter(first["x"], first["y"], c=best.reindex(first.index), s=14, cmap="viridis")
axes[1].set(xlabel="x on probe (µm)", ylabel="distance from tip (µm)", title="Probe position")
fig.colorbar(image, ax=axes[1], label="log10 best selectivity")
plt.tight_layout()

## Explorer

- **Recording**: the run used to filter and sort the units.
- **Units**: Kilosort label.
- **Min selectivity**: hide units below this selectivity in the chosen run (log slider).
- **Ignore blank**: filter and sort by `selectivity_stim` (tuning among stimuli) instead of `selectivity`.
- **Sort by**: order of the unit list.
- **Show all recordings**: plot the unit in every run side by side, or only the chosen one.

Each column shows a run: raster and PSTH grouped by condition (shaded = stimulus period), then the tuning curve
(mean rate ± SEM during the stimulus; dashed = baseline). The probe map on the right shows all units (grey),
the units passing the filters (blue) and the current unit (red star).

In [ ]:
def plot_unit_summary(unit, shown, filter_name, passing):
    """Raster, PSTH and tuning curve of one unit in each run in ``shown``, plus a probe map."""
    fig = plt.figure(figsize=(4.8 * len(shown) + 2.6, 9))
    grid = fig.add_gridspec(3, len(shown) + 1, height_ratios=[2.2, 1, 1], width_ratios=[4.8] * len(shown) + [2.6])
    for i, name in enumerate(shown):
        d = data[name]
        row = d["units"].loc[unit]
        times = d["trains"][unit]
        ax_raster = fig.add_subplot(grid[0, i])
        ax_psth = fig.add_subplot(grid[1, i], sharex=ax_raster)
        ax_tuning = fig.add_subplot(grid[2, i])
        sp.plot_unit(times, d["trials"]["onset"], d["trials"]["condition"], window=window, bin_size=0.02,
                     smooth=0.02, shade=d["stim_window"], axes=(ax_raster, ax_psth))
        ax_psth.legend(fontsize="x-small", frameon=False, ncol=2)
        ax_raster.set_title(f"{name}\nselectivity {row['selectivity']:.1f} (stimuli only {row['selectivity_stim']:.1f}), "
                            f"response {row['response']:+.1f} Hz", fontsize="small")
        rates = sp.condition_rates(times, d["trials"]["onset"], d["trials"]["condition"], d["stim_window"])
        sp.plot_tuning(ax_tuning, rates, baseline=row["baseline_rate"], color="k")
        ax_tuning.set_xlabel("condition")
    units = data[filter_name]["units"]
    ax_map = fig.add_subplot(grid[:, -1])
    ax_map.scatter(units["x"], units["y"], s=8, c="0.85")
    ax_map.scatter(passing["x"], passing["y"], s=10, c="C0")
    ax_map.scatter(units.loc[unit, "x"], units.loc[unit, "y"], s=160, c="C3", marker="*")
    ax_map.set(xlabel="x (µm)", ylabel="distance from tip (µm)", title="probe")
    label = units.loc[unit, "label"]
    fig.suptitle(f"unit {unit} ({label}), y = {units.loc[unit, 'y']:.0f} µm")
    fig.tight_layout()
    return fig


recording_w = w.Dropdown(options=list(data), description="Recording")
label_w = w.ToggleButtons(options=["all", "good", "mua"], description="Units")
min_w = w.FloatLogSlider(value=3, base=10, min=-1, max=3, step=0.05, description="Min selectivity",
                         readout_format=".1f", continuous_update=False, style={"description_width": "initial"})
blank_w = w.Checkbox(value=False, description="Ignore blank (tuning among stimuli)")
sort_w = w.Dropdown(options=["selectivity", "response", "depth", "unit number"], description="Sort by")
all_w = w.Checkbox(value=True, description="Show all recordings")
unit_w = w.Dropdown(description="Unit", layout=w.Layout(width="420px"))
previous_w = w.Button(description="◀ previous", layout=w.Layout(width="110px"))
next_w = w.Button(description="next ▶", layout=w.Layout(width="110px"))
count_w = w.HTML()
out = w.Output()


def passing_units():
    """Units of the chosen recording that pass the filters, in the chosen order."""
    units = data[recording_w.value]["units"]
    column = "selectivity_stim" if blank_w.value else "selectivity"
    keep = units[column] >= min_w.value
    if label_w.value != "all":
        keep &= units["label"] == label_w.value
    units = units[keep]
    if sort_w.value == "unit number":
        return units.sort_index(), column
    by = {"selectivity": column, "response": "response", "depth": "y"}[sort_w.value]
    return units.sort_values(by, ascending=False), column


def draw(*_):
    out.clear_output(wait=True)
    with out:
        if unit_w.value is None:
            print("No units pass the filters: lower the minimum selectivity or change the label.")
            return
        passing, _ = passing_units()
        shown = list(data) if all_w.value else [recording_w.value]
        plot_unit_summary(unit_w.value, shown, recording_w.value, passing)
        plt.show()


def refresh_units(*_):
    passing, column = passing_units()
    current = unit_w.value
    options = [(f"{u}  ({r['label']}, selectivity {r[column]:.1f}, y {r['y']:.0f} µm)", u) for u, r in passing.iterrows()]
    unit_w.unobserve(draw, "value")
    unit_w.options = options
    unit_w.value = current if current in passing.index else (options[0][1] if options else None)
    unit_w.observe(draw, "value")
    count_w.value = f"<b>{len(passing)}</b> of {len(data[recording_w.value]['units'])} units pass"
    draw()


def step(direction):
    values = [value for _, value in unit_w.options]
    if values and unit_w.value in values:
        unit_w.value = values[(values.index(unit_w.value) + direction) % len(values)]


for control in (recording_w, label_w, min_w, blank_w, sort_w):
    control.observe(refresh_units, "value")
all_w.observe(draw, "value")
unit_w.observe(draw, "value")
previous_w.on_click(lambda _: step(-1))
next_w.on_click(lambda _: step(+1))

display(w.VBox([
    w.HBox([recording_w, sort_w, all_w]),
    w.HBox([label_w, blank_w]),
    w.HBox([min_w, count_w]),
    w.HBox([unit_w, previous_w, next_w]),
    out,
]))
refresh_units()

## Table of the units passing the filters

Re-run this cell after changing the controls to list the units (and copy unit numbers into other analyses).

In [ ]:
passing, column = passing_units()
passing.round(2)